# Reference data: host → `host_group`

Build committed `data/external/host_groups.csv` used by `09_clean_viruses` (spec `04`).

| Column | Meaning |
|--------|--------|
| `host_key` | Normalized match key via `norm_key()` (unique) |
| `host_group` | Closed vocabulary **except** `unknown` |
| `notes` | Provenance tag |

**Seeds:** VectorNet `vector_taxon_groups.csv` scientific names + `GLOBAL_VECTOR_GENUS_TO_GROUP`, plus curated high-frequency vertebrate / outbreak / lab aliases from the multi-source virus extract.

**Runtime classify** (`classify_host_group`) still applies lab / arthropod / vertebrate heuristics and `isolation_source` fallback; this CSV holds exact aliases that should win first.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_reference_dir,
    find_repo_root,
    load_virus_detections,
)
from vector_guard.resolve import (
    GLOBAL_VECTOR_GENUS_TO_GROUP,
    HOST_GROUPS,
    build_host_alias_map,
    build_host_taxon_lookup,
    classify_host_group,
    load_host_groups,
    load_vector_taxon_groups,
    norm_key,
)

REPO_ROOT = find_repo_root(Path.cwd())
REF_DIR = default_reference_dir(REPO_ROOT)
REF_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = REF_DIR / "host_groups.csv"
ALLOWED = set(HOST_GROUPS) - {"unknown"}

REPO_ROOT, OUT_PATH

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/external/host_groups.csv'))

## Inventory raw virus hosts

In [2]:
raw = load_virus_detections()
assert "host" in raw.columns and "isolation_source" in raw.columns

host_counts = (
    raw["host"]
    .fillna("")
    .astype(str)
    .str.strip()
    .value_counts()
    .rename("n_rows")
    .to_frame()
    .reset_index(names="host")
)

iso_counts = (
    raw["isolation_source"]
    .fillna("")
    .astype(str)
    .str.strip()
    .value_counts()
    .rename("n_rows")
    .to_frame()
    .reset_index(names="isolation_source")
)

print(f"Raw rows: {len(raw):,}")
print(f"Distinct host: {len(host_counts):,}")
print(f"Distinct isolation_source: {len(iso_counts):,}")
display(host_counts.head(20))
display(iso_counts.head(15))

Raw rows: 168,307
Distinct host: 1,616
Distinct isolation_source: 1,828


,host,n_rows
0,Homo sapiens,82844
1,,14472
2,Domestic - Cattle,5300
3,Domestic - sheep,5018
4,Domestic - unspecified mammal,3476
5,Domestic - Sheep,2667
6,Domestic - cattle,2447
7,Ovis aries,2281
8,bovine,2280
9,sheep,2270


,isolation_source,n_rows
0,,54947
1,serum,30472
2,Domestic,24810
3,blood,11992
4,plasma,5543
5,cell supernatant,2118
6,cell culture,1671
7,whole blood,1413
8,human blood,1334
9,brain,1227


## Seed rows + curated aliases

1. VectorNet taxa (species + genus) and `GLOBAL_VECTOR_GENUS_TO_GROUP`
2. Curated vertebrate / outbreak / lab / abbreviated-vector aliases for frequent unmatched strings

In [3]:
def add_alias(
    rows: list[dict[str, str]],
    seen: dict[str, str],
    host_text: str,
    host_group: str,
    notes: str,
) -> None:
    group = host_group.casefold().strip()
    if group not in ALLOWED:
        raise ValueError(f"Invalid host_group {group!r} for {host_text!r}")
    key = norm_key(host_text)
    if not key:
        return
    existing = seen.get(key)
    if existing is not None:
        if existing != group:
            raise ValueError(
                f"Conflict for {key!r}: {existing!r} vs {group!r} ({host_text!r})"
            )
        return
    seen[key] = group
    rows.append({"host_key": key, "host_group": group, "notes": notes})


rows: list[dict[str, str]] = []
seen: dict[str, str] = {}

# --- VectorNet taxa + global genera / higher taxa ---
vector_taxa = load_vector_taxon_groups(REF_DIR)
for row in vector_taxa.itertuples(index=False):
    name = str(getattr(row, "scientific_name", "")).strip()
    group = str(getattr(row, "vector_group", "")).strip()
    if not name or not group:
        continue
    add_alias(rows, seen, name, group, "vectornet_taxon")
    genus = name.split()[0]
    add_alias(rows, seen, genus, group, f"vectornet_genus:{genus}")

for genus, group in sorted(GLOBAL_VECTOR_GENUS_TO_GROUP.items()):
    add_alias(rows, seen, genus, group, "global_vector_genus")

print(f"After vector seeds: {len(rows):,} rows")

After vector seeds: 251 rows


In [4]:
# Curated aliases: (raw host string, host_group, notes)
# Prefer exact spellings seen in virus_detections.tsv; heuristics still cover many others.
CURATED_ALIASES: list[tuple[str, str, str]] = [
    # Human / typos
    ("Homo sapiens", "human", "curated_vertebrate"),
    ("homo sapiens", "human", "curated_vertebrate"),
    ("Human", "human", "curated_vertebrate"),
    ("human", "human", "curated_vertebrate"),
    ("Homos sapiens", "human", "curated_vertebrate"),
    ("Home sapiens: Female", "human", "curated_vertebrate"),
    ("Home sapiens: Male", "human", "curated_vertebrate"),
    # ECDC Atlas tokens (after :count strip at classify time; store base forms)
    ("human", "human", "ecdc_token"),
    ("bird_outbreaks", "bird", "ecdc_token"),
    ("equid_outbreaks", "livestock", "ecdc_token"),
    # EMPRES-i / domestic livestock
    ("Domestic - Cattle", "livestock", "curated_vertebrate"),
    ("Domestic - cattle", "livestock", "curated_vertebrate"),
    ("Domestic - Sheep", "livestock", "curated_vertebrate"),
    ("Domestic - sheep", "livestock", "curated_vertebrate"),
    ("Domestic - Goats", "livestock", "curated_vertebrate"),
    ("Domestic - Goat/sheep", "livestock", "curated_vertebrate"),
    ("Domestic - Equidae", "livestock", "curated_vertebrate"),
    ("Domestic - horse", "livestock", "curated_vertebrate"),
    ("Domestic - Horse", "livestock", "curated_vertebrate"),
    ("Domestic - Swine", "livestock", "curated_vertebrate"),
    ("Domestic - Small Ruminant", "livestock", "curated_vertebrate"),
    ("Domestic - unspecified mammal", "livestock", "curated_vertebrate"),
    ("Domestic - Camelidae (unidentified)", "livestock", "curated_vertebrate"),
    ("bovine", "livestock", "curated_vertebrate"),
    ("Bovine", "livestock", "curated_vertebrate"),
    ("cattle", "livestock", "curated_vertebrate"),
    ("sheep", "livestock", "curated_vertebrate"),
    ("ovine", "livestock", "curated_vertebrate"),
    ("goat", "livestock", "curated_vertebrate"),
    ("horse", "livestock", "curated_vertebrate"),
    ("equine", "livestock", "curated_vertebrate"),
    ("swine", "livestock", "curated_vertebrate"),
    ("Ruminant", "livestock", "curated_vertebrate"),
    ("Ovis aries", "livestock", "curated_vertebrate"),
    ("Ovis orientalis aries", "livestock", "curated_vertebrate"),
    ("Bos taurus", "livestock", "curated_vertebrate"),
    ("Bubalus bubalis", "livestock", "curated_vertebrate"),
    ("Capra hircus", "livestock", "curated_vertebrate"),
    ("Equus caballus", "livestock", "curated_vertebrate"),
    ("Sus scrofa", "livestock", "curated_vertebrate"),
    ("Vicugna pacos", "livestock", "curated_vertebrate"),
    ("Bovidae", "livestock", "curated_vertebrate"),
    ("lamb", "livestock", "curated_vertebrate"),
    ("bison", "livestock", "curated_vertebrate"),
    ("dog", "livestock", "curated_vertebrate"),
    ("canine", "livestock", "curated_vertebrate"),
    ("Canis familiaris", "livestock", "curated_vertebrate"),
    ("Canis lupus familiaris", "livestock", "curated_vertebrate"),
    ("Canis lupus familaris", "livestock", "curated_vertebrate"),
    ("cat", "livestock", "curated_vertebrate"),
    # Birds (common names / taxa heuristics may miss)
    ("American Crow", "bird", "curated_vertebrate"),
    ("American crow", "bird", "curated_vertebrate"),
    ("Western Scrub Jay", "bird", "curated_vertebrate"),
    ("Western Scrub/Jay", "bird", "curated_vertebrate"),
    ("California Scrub-Jay", "bird", "curated_vertebrate"),
    ("House Finch", "bird", "curated_vertebrate"),
    ("blackbird", "bird", "curated_vertebrate"),
    ("goshawk", "bird", "curated_vertebrate"),
    ("Crow", "bird", "curated_vertebrate"),
    ("free-ranging wild birds", "bird", "curated_vertebrate"),
    ("Wild - Unspecified bird", "bird", "curated_vertebrate"),
    ("Wild - Carrion crow (Corvus corone)", "bird", "curated_vertebrate"),
    ("Wild - Eurasian jackdaw (Coloeus monedula)", "bird", "curated_vertebrate"),
    ("Wild - Magpie", "bird", "curated_vertebrate"),
    ("Wild - Hooded Crow (Corvus cornix)", "bird", "curated_vertebrate"),
    ("Wild - Accipitridae (incognita)", "bird", "curated_vertebrate"),
    ("Turdus merula", "bird", "curated_vertebrate"),
    ("Corvus brachyrhynchos", "bird", "curated_vertebrate"),
    ("Cyanocitta cristata", "bird", "curated_vertebrate"),
    ("Astur gentilis", "bird", "curated_vertebrate"),
    ("Accipiter gentilis", "bird", "curated_vertebrate"),
    ("Bubo virginianus", "bird", "curated_vertebrate"),
    ("Bubo scandiacus", "bird", "curated_vertebrate"),
    ("Strix nebulosa", "bird", "curated_vertebrate"),
    ("Cyanistes caeruleus", "bird", "curated_vertebrate"),
    ("Garrulus glandarius", "bird", "curated_vertebrate"),
    ("Columba palumbus", "bird", "curated_vertebrate"),
    ("Haemorhous mexicanus", "bird", "curated_vertebrate"),
    ("Aphelocoma californica", "bird", "curated_vertebrate"),
    ("Sternula antillarum", "bird", "curated_vertebrate"),
    ("Alectoris rufa", "bird", "curated_vertebrate"),
    ("Pelecanus erythrorhynchos", "bird", "curated_vertebrate"),
    ("Pica pica", "bird", "curated_vertebrate"),
    ("Baeolophus bicolor", "bird", "curated_vertebrate"),
    ("Poecile carolinensis", "bird", "curated_vertebrate"),
    ("Buteo lineatus", "bird", "curated_vertebrate"),
    # Wildlife mammals
    ("Odocoileus virginianus", "wildlife_mammal", "curated_vertebrate"),
    ("white-tailed deer", "wildlife_mammal", "curated_vertebrate"),
    ("white-tail deer", "wildlife_mammal", "curated_vertebrate"),
    ("mule deer", "wildlife_mammal", "curated_vertebrate"),
    ("Cervidae sp.", "wildlife_mammal", "curated_vertebrate"),
    ("cervine", "wildlife_mammal", "curated_vertebrate"),
    ("Reindeer", "wildlife_mammal", "curated_vertebrate"),
    ("Mazama nana", "wildlife_mammal", "curated_vertebrate"),
    ("Alouatta sp.", "wildlife_mammal", "curated_vertebrate"),
    ("Alouatta", "wildlife_mammal", "curated_vertebrate"),
    ("Alouatta guariba clamitans", "wildlife_mammal", "curated_vertebrate"),
    ("Macaca mulatta", "wildlife_mammal", "curated_vertebrate"),
    ("Callithrix", "wildlife_mammal", "curated_vertebrate"),
    ("Callithrix sp.", "wildlife_mammal", "curated_vertebrate"),
    ("Callithrix penicillata", "wildlife_mammal", "curated_vertebrate"),
    ("Cebidae", "wildlife_mammal", "curated_vertebrate"),
    ("Chlorocebus aethiops", "wildlife_mammal", "curated_vertebrate"),
    ("non-human primate", "wildlife_mammal", "curated_vertebrate"),
    ("Mus musculus", "wildlife_mammal", "curated_vertebrate"),
    ("mice", "wildlife_mammal", "curated_vertebrate"),
    ("Cricetulus griseus", "wildlife_mammal", "curated_vertebrate"),
    ("Mesocricetus auratus", "wildlife_mammal", "curated_vertebrate"),
    ("Ondatra zibethicus", "wildlife_mammal", "curated_vertebrate"),
    ("Myodes glareolus", "wildlife_mammal", "curated_vertebrate"),
    ("Microtus obscurus", "wildlife_mammal", "curated_vertebrate"),
    ("Clethrionomys rufocanus", "wildlife_mammal", "curated_vertebrate"),
    ("Apodemus agrarius", "wildlife_mammal", "curated_vertebrate"),
    ("Rhizomys pruinosus", "wildlife_mammal", "curated_vertebrate"),
    ("Sciurus carolinensis", "wildlife_mammal", "curated_vertebrate"),
    ("Glossophaga mutica", "wildlife_mammal", "curated_vertebrate"),
    ("pangolin", "wildlife_mammal", "curated_vertebrate"),
    ("Canis latrans", "wildlife_mammal", "curated_vertebrate"),
    ("Canis lupus", "wildlife_mammal", "curated_vertebrate"),
    # Lab / culture / strain material
    ("Vero-E6", "lab", "lab_line"),
    ("VeroE6", "lab", "lab_line"),
    ("Vero", "lab", "lab_line"),
    ("Vero cell", "lab", "lab_line"),
    ("Vero cells", "lab", "lab_line"),
    ("vero cells", "lab", "lab_line"),
    ("C6/36", "lab", "lab_line"),
    ("C6/36 cells", "lab", "lab_line"),
    ("C6/36 Cells", "lab", "lab_line"),
    ("BHK-21 cells", "lab", "lab_line"),
    ("Bhk-21", "lab", "lab_line"),
    ("BHK-BSR cell", "lab", "lab_line"),
    ("ECE BHK3", "lab", "lab_line"),
    ("cDNA clone", "lab", "lab_line"),
    ("CBER strain", "lab", "lab_line"),
    ("BEI strain", "lab", "lab_line"),
    ("Aedes aegypti cells", "lab", "lab_line"),
    ("Homo sapiens cells", "lab", "lab_line"),
    ("Medium Vero infected cells", "lab", "lab_line"),
    ("supernatants of C6/36 cells", "lab", "lab_line"),
    # Abbreviated / misspelled mosquito vectors
    ("Cx. quinquefasciatus", "mosquito", "curated_vector_abbrev"),
    ("Cx pipiens", "mosquito", "curated_vector_abbrev"),
    ("Cx Pipiens", "mosquito", "curated_vector_abbrev"),
    ("Culix quinquefasciatus", "mosquito", "curated_vector_abbrev"),
    ("Ae. aegypti", "mosquito", "curated_vector_abbrev"),
    ("Ae. albopictus", "mosquito", "curated_vector_abbrev"),
    ("Culex cf. pipiens/torrentium", "mosquito", "curated_vector_abbrev"),
    ("Culex pipiens s.l./Culex torrentium", "mosquito", "curated_vector_abbrev"),
    ("Haemagogus leucocelaenus", "mosquito", "global_vector_genus"),
    ("Haemagogus janthinomys", "mosquito", "global_vector_genus"),
    ("Hg. janthinomys", "mosquito", "curated_vector_abbrev"),
    ("Hg. leucocelaenus", "mosquito", "curated_vector_abbrev"),
    ("Haemagogus", "mosquito", "global_vector_genus"),
    # Tick typos / extras
    ("Ixodus persulcatus", "tick", "curated_vector_abbrev"),
    ("Dermacetor reticulatus", "tick", "curated_vector_abbrev"),
    ("Tick", "tick", "curated_vector_abbrev"),
    ("tick", "tick", "curated_vector_abbrev"),
    ("mosquito", "mosquito", "curated_vector_abbrev"),
    # Extra frequent residuals from inventory QA
    ("Cornix cornix", "bird", "curated_vertebrate"),
    ("Accipiter striatus", "bird", "curated_vertebrate"),
    ("Phoenicopterus chilensis", "bird", "curated_vertebrate"),
    ("Regulus regulus", "bird", "curated_vertebrate"),
    ("Myotis davidii", "wildlife_mammal", "curated_vertebrate"),
    ("Artibeus jamaicensis", "wildlife_mammal", "curated_vertebrate"),
    ("Molossus sinaloae", "wildlife_mammal", "curated_vertebrate"),
    ("Clethrionomys rutilus", "wildlife_mammal", "curated_vertebrate"),
    ("BSR2", "lab", "lab_line"),
]

for host_text, group, notes in CURATED_ALIASES:
    add_alias(rows, seen, host_text, group, notes)

print(f"After curated aliases: {len(rows):,} rows")
pd.Series({g: sum(1 for r in rows if r["host_group"] == g) for g in sorted(ALLOWED)}, name="n_keys")

After curated aliases: 403 rows


bird                39
biting_midge        82
human                5
lab                 19
livestock           37
mosquito           120
sandfly             43
tick                21
wildlife_mammal     37
Name: n_keys, dtype: int64

## Write CSV + reload checks

In [5]:
out = (
    pd.DataFrame(rows)
    .sort_values(["host_group", "host_key"])
    .reset_index(drop=True)
)
assert list(out.columns) == ["host_key", "host_group", "notes"]
assert out["host_key"].ne("").all()
assert not out["host_key"].duplicated().any()
assert set(out["host_group"]) <= ALLOWED
assert "unknown" not in set(out["host_group"])

out.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
display(out.groupby("host_group").size().rename("n_keys").reset_index())
display(out.head(12))
display(out.tail(8))

Wrote /workspaces/vector-guard/data/external/host_groups.csv (403 rows)


,host_group,n_keys
0,bird,39
1,biting_midge,82
2,human,5
3,lab,19
4,livestock,37
5,mosquito,120
6,sandfly,43
7,tick,21
8,wildlife_mammal,37


,host_key,host_group,notes
0,accipiter-gentilis,bird,curated_vertebrate
1,accipiter-striatus,bird,curated_vertebrate
2,alectoris-rufa,bird,curated_vertebrate
3,american-crow,bird,curated_vertebrate
4,aphelocoma-californica,bird,curated_vertebrate
5,astur-gentilis,bird,curated_vertebrate
6,baeolophus-bicolor,bird,curated_vertebrate
7,bird-outbreaks,bird,ecdc_token
8,blackbird,bird,curated_vertebrate
9,bubo-scandiacus,bird,curated_vertebrate


,host_key,host_group,notes
395,odocoileus-virginianus,wildlife_mammal,curated_vertebrate
396,ondatra-zibethicus,wildlife_mammal,curated_vertebrate
397,pangolin,wildlife_mammal,curated_vertebrate
398,reindeer,wildlife_mammal,curated_vertebrate
399,rhizomys-pruinosus,wildlife_mammal,curated_vertebrate
400,sciurus-carolinensis,wildlife_mammal,curated_vertebrate
401,white-tail-deer,wildlife_mammal,curated_vertebrate
402,white-tailed-deer,wildlife_mammal,curated_vertebrate


In [6]:
reloaded = load_host_groups(REF_DIR)
alias_map = build_host_alias_map(reloaded)
taxon_lookup = build_host_taxon_lookup(vector_taxa)

assert list(reloaded.columns) == ["host_key", "host_group", "notes"]
assert len(alias_map) == len(reloaded)

# Coverage over raw host strings
host_series = raw["host"].fillna("").astype(str)
classified = host_series.map(
    lambda h: classify_host_group(
        h, host_alias_map=alias_map, taxon_to_group=taxon_lookup, isolation_source=""
    )
)
counts = classified.value_counts()
non_empty_mask = host_series.str.strip().ne("") & ~host_series.str.strip().str.casefold().isin(
    {"missing", "unknown", "na", "n/a", "none", "nan"}
)
resolved_non_empty = (classified.loc[non_empty_mask] != "unknown").mean()

print("host_group counts (host-only, no isolation_source fallback):")
display(counts.rename("n_rows").to_frame())
print(
    f"Non-empty host resolved share: {resolved_non_empty:.3%} "
    f"({int((classified.loc[non_empty_mask] != 'unknown').sum()):,} / {int(non_empty_mask.sum()):,})"
)

still_unknown = (
    host_counts.assign(
        host_group=lambda d: d["host"].map(
            lambda h: classify_host_group(
                h, host_alias_map=alias_map, taxon_to_group=taxon_lookup
            )
        )
    )
    .loc[lambda d: (d["host_group"] == "unknown") & d["host"].str.strip().ne("")]
    .sort_values("n_rows", ascending=False)
)
print(f"Distinct non-empty hosts still unknown: {len(still_unknown):,}")
display(still_unknown.head(25))

# Spot-check locked edge cases (with isolation_source where relevant)
checks = [
    ("Homo sapiens", "passage 3 in Aedes albopictus C6/36", "human"),
    ("Aedes aegypti cells", "", "lab"),
    ("Vero-E6", "", "lab"),
    ("Culex pipiens", "", "mosquito"),
    ("Haemaphysalis longicornis", "", "tick"),
    ("Ixodoidea", "", "tick"),
    ("Wild - Carrion crow (Corvus corone)", "", "bird"),
    ("Domestic - Cattle | Domestic - Sheep", "", "livestock"),
    ("Domestic - unspecified mammal", "", "livestock"),
    ("equid_outbreaks:1;bird_outbreaks:2", "", "bird"),
    ("Cx. quinquefasciatus", "", "mosquito"),
    ("", "mosquito pool", "mosquito"),
    ("", "I. persulcatus", "tick"),
    ("", "primary cell culture from Aedes", "lab"),
    ("", "serum", "unknown"),
]
for host, iso, expect in checks:
    got = classify_host_group(
        host, host_alias_map=alias_map, taxon_to_group=taxon_lookup, isolation_source=iso
    )
    assert got == expect, (host, iso, got, expect)

print("OK — host_groups.csv ready for 09_clean_viruses.")

host_group counts (host-only, no isolation_source fallback):


,n_rows
host,
human,83876
livestock,40477
unknown,17142
mosquito,11037
tick,6422
bird,5060
wildlife_mammal,2962
lab,604
biting_midge,477


Non-empty host resolved share: 98.953% (151,165 / 152,764)
Distinct non-empty hosts still unknown: 541


,host,n_rows,host_group
23,missing,865,unknown
42,Serum,259,unknown
75,unknown,107,unknown
80,nan,99,unknown
136,Apis mellifera,47,unknown
342,Denv4,11,unknown
352,West Nile virus,11,unknown
361,Rana ridibunda,10,unknown
374,Barbarine ewe,10,unknown
378,pronghorn antelope,10,unknown


OK — host_groups.csv ready for 09_clean_viruses.
